In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import LSTM, Dense, TimeDistributed, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# ==========================================================
# חובה לשנות כאן:
# יש להכניס את הנתיב האמיתי לתיקייה הראשית של הדאטא במחשב שבו מריצים את המחברת.
# התיקייה הראשית צריכה להכיל בתוכה את תיקיות המחלקות, למשל:
# Dataset/
#   Violent/
#   NonViolent/
#
# או:
# Dataset/
#   violent_5sec_clips/
#   Normal_5sec_clips/
# ==========================================================

DATASET_PATH = input("Enter the full path to the dataset main folder: ").strip().strip('"').strip("'")

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError("The dataset path does not exist. Please check the path and run this cell again.")

print("Dataset path was found successfully:")
print(DATASET_PATH)

NameError: name 'files' is not defined

In [ ]:
# התוצאות יישמרו בתיקייה מקומית ליד המחברת / בסביבת ההרצה הנוכחית
PROJECT_PATH = os.getcwd()

MODEL_DIR = os.path.join(PROJECT_PATH, "trained_model")
RESULTS_DIR = os.path.join(PROJECT_PATH, "results")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Project path:")
print(PROJECT_PATH)

print("Model directory:")
print(MODEL_DIR)

print("Results directory:")
print(RESULTS_DIR)

In [ ]:
def extract_frames(video_path, num_frames=20, img_size=224):
    frames = []

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return frames

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return frames

    skip_interval = max(int(total_frames / num_frames), 1)

    for i in range(num_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * skip_interval)

        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.resize(frame, (img_size, img_size))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        frames.append(frame)

    cap.release()

    return frames

In [ ]:
video_extensions = ('.avi', '.mp4', '.mov', '.mkv')

files = []

for root, dirs, filenames in os.walk(DATASET_PATH):
    for filename in filenames:
        if filename.lower().endswith(video_extensions):
            full_path = os.path.join(root, filename)
            files.append(full_path)

print(f"Total videos found: {len(files)}")

if len(files) == 0:
    raise ValueError("No video files were found. Please check the dataset folder structure.")

In [ ]:
print("Checking first video:")
print(files[0])

test_frames = extract_frames(files[0])

if len(test_frames) > 0:
    print(f"Success! Extracted {len(test_frames)} frames.")

    plt.imshow(test_frames[0])
    plt.axis('off')
    plt.show()
else:
    raise ValueError("Could not extract frames from the first video.")

In [ ]:
# ==========================================================
# חובה לבדוק ולעדכן במידת הצורך:
# כאן כותבים את שמות התיקיות שנחשבות לאלימות.
# כל תיקייה שמופיעה כאן תקבל label = 1.
# כל תיקייה אחרת תקבל label = 0.
# ==========================================================

violence_folders = [
    "Violent",
    "violent_5sec_clips",
    "Fighting",
    "Fight",
    "Shooting"
]

In [ ]:
data = []
labels = []

for filepath in files:
    folder_name = os.path.basename(os.path.dirname(filepath))

    data.append(filepath)

    if folder_name in violence_folders:
        labels.append(1)
    else:
        labels.append(0)

print("Finished creating data and labels.")
print(f"Total samples: {len(data)}")

In [ ]:
all_folders = sorted(set(os.path.basename(os.path.dirname(path)) for path in data))

print("Folders found in dataset:")
for folder in all_folders:
    print(folder)

violence_count = sum(labels)
normal_count = len(labels) - violence_count

print("\nLabel summary:")
print(f"Violence samples label=1: {violence_count}")
print(f"Normal samples label=0: {normal_count}")

if violence_count == 0:
    raise ValueError("No violence samples were labeled as 1. Check violence_folders names.")

if normal_count == 0:
    raise ValueError("No normal samples were labeled as 0. Check violence_folders names.")

In [ ]:
train_data, val_data, train_labels, val_labels = train_test_split(
    data,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Train / Validation split completed.")
print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

print(f"\nTrain violence samples: {sum(train_labels)}")
print(f"Train normal samples: {len(train_labels) - sum(train_labels)}")

print(f"\nValidation violence samples: {sum(val_labels)}")
print(f"Validation normal samples: {len(val_labels) - sum(val_labels)}")

In [ ]:
class VideoDataGenerator(Sequence):
    def __init__(self, sample_list, labels, batch_size=8, num_frames=20, img_size=224, shuffle=True):
        self.sample_list = np.array(sample_list)
        self.labels = np.array(labels)
        self.batch_size = batch_size
        self.num_frames = num_frames
        self.img_size = img_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.sample_list))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.sample_list) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size : (index + 1) * self.batch_size]

        batch_samples = self.sample_list[batch_indexes]
        batch_labels = self.labels[batch_indexes]

        X = []

        for video_path in batch_samples:
            frames = extract_frames(
                video_path,
                num_frames=self.num_frames,
                img_size=self.img_size
            )

            while len(frames) < self.num_frames:
                empty_frame = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
                frames.append(empty_frame)

            frames = frames[:self.num_frames]

            X.append(frames)

        X = np.array(X, dtype=np.float32) / 255.0
        y = np.array(batch_labels, dtype=np.float32)

        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

In [ ]:
BATCH_SIZE = 8
NUM_FRAMES = 20
IMG_SIZE = 224

train_gen = VideoDataGenerator(
    train_data,
    train_labels,
    batch_size=BATCH_SIZE,
    num_frames=NUM_FRAMES,
    img_size=IMG_SIZE,
    shuffle=True
)

val_gen = VideoDataGenerator(
    val_data,
    val_labels,
    batch_size=BATCH_SIZE,
    num_frames=NUM_FRAMES,
    img_size=IMG_SIZE,
    shuffle=False
)

print("Generators created successfully.")
print(f"Train batches: {len(train_gen)}")
print(f"Validation batches: {len(val_gen)}")

In [ ]:
video_input = Input(shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3))

resnet_base = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)

resnet_base.trainable = False

x = TimeDistributed(resnet_base)(video_input)

x = LSTM(64)(x)

x = Dropout(0.5)(x)

x = Dense(32, activation="relu")(x)

predictions = Dense(1, activation="sigmoid")(x)

model = Model(inputs=video_input, outputs=predictions)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
EPOCHS = 5

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS
)

In [ ]:
train_accuracy = history.history["accuracy"][-1]
train_loss = history.history["loss"][-1]

val_accuracy = history.history["val_accuracy"][-1]
val_loss = history.history["val_loss"][-1]

print("Baseline Training Results:")
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Train Loss: {train_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Loss: {val_loss:.4f}")

In [ ]:
y_true = []
y_pred = []

for i in range(len(val_gen)):
    X_batch, y_batch = val_gen[i]

    predictions = model.predict(X_batch)

    predicted_labels = (predictions > 0.5).astype(int).flatten()

    y_true.extend(y_batch)
    y_pred.extend(predicted_labels)

y_true = np.array(y_true).astype(int)
y_pred = np.array(y_pred).astype(int)

baseline_accuracy = accuracy_score(y_true, y_pred)
baseline_precision = precision_score(y_true, y_pred, zero_division=0)
baseline_recall = recall_score(y_true, y_pred, zero_division=0)
baseline_f1 = f1_score(y_true, y_pred, zero_division=0)

cm = confusion_matrix(y_true, y_pred)

print("Baseline Validation Metrics:")
print(f"Accuracy: {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall: {baseline_recall:.4f}")
print(f"F1 Score: {baseline_f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Normal", "Violence"],
        zero_division=0
    )
)

In [ ]:
confusion_matrix_path = os.path.join(RESULTS_DIR, "baseline_confusion_matrix.png")

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal", "Violence"]
)

disp.plot(values_format="d")

plt.title("Baseline Confusion Matrix")
plt.savefig(confusion_matrix_path, dpi=300, bbox_inches="tight")
plt.show()

print("Confusion Matrix image saved successfully:")
print(confusion_matrix_path)

In [ ]:
accuracy_plot_path = os.path.join(RESULTS_DIR, "baseline_accuracy.png")

plt.figure()
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Baseline Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.savefig(accuracy_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Accuracy plot saved successfully:")
print(accuracy_plot_path)

In [ ]:
loss_plot_path = os.path.join(RESULTS_DIR, "baseline_loss.png")

plt.figure()
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Baseline Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig(loss_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Loss plot saved successfully:")
print(loss_plot_path)

In [ ]:
model_file_path = os.path.join(MODEL_DIR, "baseline_violence_model.h5")

model.save(model_file_path)

print("Model saved successfully:")
print(model_file_path)

In [ ]:
results_file_path = os.path.join(RESULTS_DIR, "baseline_results.txt")

with open(results_file_path, "w", encoding="utf-8") as f:
    f.write("Baseline Model Results\n")
    f.write("======================\n\n")

    f.write(f"Train Accuracy: {train_accuracy:.4f}\n")
    f.write(f"Train Loss: {train_loss:.4f}\n")
    f.write(f"Validation Accuracy: {val_accuracy:.4f}\n")
    f.write(f"Validation Loss: {val_loss:.4f}\n\n")

    f.write("Validation Metrics\n")
    f.write("------------------\n")
    f.write(f"Accuracy: {baseline_accuracy:.4f}\n")
    f.write(f"Precision: {baseline_precision:.4f}\n")
    f.write(f"Recall: {baseline_recall:.4f}\n")
    f.write(f"F1 Score: {baseline_f1:.4f}\n\n")

    f.write("Confusion Matrix:\n")
    f.write(str(cm))
    f.write("\n\n")

    f.write("Classification Report:\n")
    f.write(
        classification_report(
            y_true,
            y_pred,
            target_names=["Normal", "Violence"],
            zero_division=0
        )
    )

print("Results saved successfully:")
print(results_file_path)

In [ ]:
if baseline_f1 < 0.7:
    baseline_summary = (
        "The baseline model was not sufficient because its F1 score was relatively low, "
        "which means it did not balance well between detecting violent videos and avoiding false predictions."
    )

elif val_accuracy < 0.7:
    baseline_summary = (
        "The baseline model was not sufficient because its validation accuracy was relatively low, "
        "indicating limited generalization to unseen videos."
    )

elif abs(train_accuracy - val_accuracy) > 0.15:
    baseline_summary = (
        "The baseline model was not sufficient because there was a noticeable gap between training accuracy "
        "and validation accuracy, which may indicate overfitting or instability."
    )

else:
    baseline_summary = (
        "Although the baseline model achieved reasonable results, it was still limited because it relies on "
        "frame-level visual features and may not fully capture complex motion patterns over time."
    )

print("Baseline Summary:")
print(baseline_summary)

In [ ]:
with open(results_file_path, "a", encoding="utf-8") as f:
    f.write("\n\nBaseline Summary\n")
    f.write("================\n")
    f.write(baseline_summary)

print("Baseline summary was added to the results file.")